# Tracing + PHI redaction demo

Companion notebook for post #5. Everything here imports from the repo modules —
the real logic lives in `discharge_agent/tracing/` and `discharge_agent/redaction/`.

Prereqs: the Langfuse stack is up (`docker compose -f docker/langfuse/docker-compose.yml up -d`),
Ollama is serving `gpt-oss:20b`, and `.env` has the `LANGFUSE_*` keys.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from discharge_agent import pipeline

notes = pipeline.load_notes()
note = notes[notes["noteid"] == 1].iloc[0]
print(note["note_text"][:400])

## 1. Run one note, traced, no redaction

One root `invoke_agent` span, one generation span per LLM call, one tool span
per tool call. Open the trace in the UI at http://127.0.0.1:3000 — the span
payloads contain the raw (synthetic) PHI.

In [ ]:
from discharge_agent.tracing.instrumentation import traced_run_note, get_langfuse

out = traced_run_note(note["note_text"], note_id=1, redaction=False)
get_langfuse(redaction=False).flush()
out["decision"]

## 2. Same note, redaction ON

The Langfuse client is constructed with a `mask` callback
(`discharge_agent.redaction.masker.TraceMasker`): every span payload is walked
in-process and PHI is replaced with typed, per-trace-consistent placeholders
(`<PERSON_1>`, `<DATE_2>`, `<MRN_1>`) *before* the SDK exports anything.
The model still receives the full note — only the telemetry is masked.

In [ ]:
out_masked = traced_run_note(note["note_text"], note_id=1, redaction=True)
get_langfuse(redaction=True).flush()
out_masked["decision"]

## 3. What the masker does, standalone

In [ ]:
from discharge_agent.redaction.masker import get_masker, begin_trace_scope

scope = begin_trace_scope()  # placeholder indices are per-trace
masker = get_masker()
print(masker.mask(data={
    "discharge_date": "2024-03-18",
    "appts": [{"date": "03/25/2024", "with": "Dr. Maria Martinez"}],
    "wbc": 12.5,   # numbers pass through — that's the structural redaction
}))

## 4. Redaction policy v2 (the fix iteration)

Clinical-eponym allowlist, benign-date filters, structural recognizers for the
note template, and per-trace **date-shift surrogates** — intervals survive, so
date arithmetic in tool calls stays debuggable. See
`results/iteration_comparison.md` for before/after numbers.

In [ ]:
from discharge_agent.redaction.masker import begin_trace_scope, get_masker
from discharge_agent.redaction.policy import clinical_policy_v2

scope = begin_trace_scope()
masker_v2 = get_masker(policy=clinical_policy_v2())
print(masker_v2.mask(data=(
    "Name: PARKINSON, LISA    Unit No: 4859302\n"
    "72-year-old F with Parkinson's disease. Discharge Date: 2025-03-18.\n"
    "Follow up Dr. Chen on 03/25/2025. Pain 8/10."
)))

## 5. Fetch a redacted trace back via the API

The experiments never screenshot-scrape: traces are exported as JSON
(`experiments/fetch_traces.py`) so the debuggability tasks
(`results/debuggability_matrix.md`) are reproducible and diffable.

In [ ]:
import json, subprocess, glob
subprocess.run(["python", "-m", "experiments.fetch_traces",
                "--tag", "redaction:on", "--out", "results/traces", "--limit", "3"], check=True)
path = sorted(glob.glob("results/traces/note_1_*.json"))[-1]
trace = json.load(open(path))
[(o["type"], o["name"]) for o in trace["observations"]]